In [6]:
!pip install -q datasets scikit-learn

import re, numpy as np, pandas as pd, torch, torch.nn as nn
from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ds = load_dataset("Yelp/yelp_review_full")
train = pd.DataFrame(ds["train"].shuffle(seed=42).select(range(20000)))
test = pd.DataFrame(ds["test"].shuffle(seed=42).select(range(5000)))
tr, val = train_test_split(train, test_size=0.1, random_state=42, stratify=train["label"])

tok = lambda s: re.findall(r"[a-z0-9']+|[!?.]", s.lower())
counter = Counter(w for t in tr["text"] for w in tok(t))
vocab = ["<pad>", "<unk>"] + [w for w, c in counter.items() if c >= 3]
stoi = {w: i for i, w in enumerate(vocab)}
print("Vocab size:", len(vocab))

MAXLEN = 200
def encode(t):
    ids = [stoi.get(w, 1) for w in tok(t)][:MAXLEN]
    return ids + [0] * (MAXLEN - len(ids))

def make(df):
    X = torch.tensor([encode(t) for t in df["text"]], dtype=torch.long)
    y = torch.tensor(df["label"].values, dtype=torch.long)
    return X, y

X_tr, y_tr = make(tr)
X_val, y_val = make(val)
X_te, y_te = make(test)
print(X_tr.shape, X_val.shape, X_te.shape)

Device: cuda
Vocab size: 18637
torch.Size([18000, 200]) torch.Size([2000, 200]) torch.Size([5000, 200])


In [7]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb=128, hid=128, n_classes=5, dropout=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hid * 2, n_classes)

    def forward(self, x):
        mask = (x != 0).unsqueeze(-1)
        out, _ = self.lstm(self.emb(x))
        out = out.masked_fill(~mask, -1e4)             # smaller fill value
        pooled = out.max(1).values
        has_tokens = mask.any(1)                        # False for fully empty reviews
        pooled = torch.where(has_tokens, pooled, torch.zeros_like(pooled))
        return self.fc(self.drop(pooled))

model = LSTMClassifier(len(vocab)).to(device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Parameters: 2651013


In [4]:
from torch.utils.data import DataLoader, TensorDataset
import copy

def predict(X, batch=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            preds.append(model(X[i:i + batch].to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy()

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_f1, best_state = 0, None
for epoch in range(6):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * len(xb)
    f1 = f1_score(y_val.numpy(), predict(X_val), average="macro")
    print(f"Epoch {epoch+1}: train loss {total/len(X_tr):.4f} | val macro F1 {f1:.4f}")
    if f1 > best_f1:
        best_f1, best_state = f1, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)   # keep the best epoch, not the last

Epoch 1: train loss 1.4349 | val macro F1 0.4687
Epoch 2: train loss 1.1693 | val macro F1 0.5018
Epoch 3: train loss 1.0239 | val macro F1 0.5127
Epoch 4: train loss 0.9283 | val macro F1 0.5427
Epoch 5: train loss 0.8399 | val macro F1 0.5267
Epoch 6: train loss 0.7637 | val macro F1 0.5475


<All keys matched successfully>

In [5]:
test_pred = predict(X_te)
print("LSTM Accuracy:", accuracy_score(y_te.numpy(), test_pred))
print("LSTM Macro F1:", f1_score(y_te.numpy(), test_pred, average="macro"))

LSTM Accuracy: 0.5514
LSTM Macro F1: 0.5513261901545704
